<a href="https://colab.research.google.com/github/ksehar99/DL_Blockchain_Pipeline_For_DR_EarlyDiagnosis/blob/main/DR_Blockchain_Web3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sub-Pipeline 3 - Blockchain Integration
## DL + Blockchain Pipeline for Diabetic Retinopathy Diagnosis
### Web3.py - Patient Registration, Diagnosis Upload & Verification on Sepolia Testnet

In [1]:
# ─── Setup & Imports ────────────────────────────────────────────────────────

# Mount Google Drive — dataset and patient JSON are stored persistently here
from google.colab import drive
drive.mount('/content/drive')

# Kaggle API credentials — required to download the APTOS 2019 dataset
import os
os.environ['KAGGLE_USERNAME'] = 'khadija4058'
os.environ['KAGGLE_KEY'] = 'KGAT_61e373e830995974eee3022ca7f6673f'

# Dataset download — run only once, stored in Drive to persist across sessions
# !kaggle datasets download -d mariaherrerot/aptos2019
# !unzip aptos2019.zip -d /content/drive/MyDrive/aptos2019

# Install required libraries
!pip install web3    # For connecting to Ethereum blockchain via Web3.py
!pip install faker   # For generating synthetic patient data

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.5/587.5 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.0/344.0 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 12.9 MB/s eta 0:00:00


## Connecting with Sepolia Testnet

In [2]:
# ─── Blockchain Connection ───────────────────────────────────────────────────

from web3 import Web3
from google.colab import userdata

# Load sensitive credentials from Colab Secrets (never hardcode keys)
ALCHEMY_URL = userdata.get('ALCHEMY_URL')       # Alchemy RPC endpoint for Sepolia
PRIVATE_KEY = userdata.get('MMprivatekey')       # MetaMask wallet private key

# Connect to Sepolia testnet via Alchemy
w3 = Web3(Web3.HTTPProvider(ALCHEMY_URL))
print("Connected:", w3.is_connected())

# Derive owner wallet address from private key
account = w3.eth.account.from_key(PRIVATE_KEY)
owner_address = account.address
print("Owner Address:", owner_address)

# ─── Smart Contract Setup ────────────────────────────────────────────────────

# Deployed contract address on Sepolia testnet
CONTRACT_ADDRESS = "0x1B48C70996C099B1fb961fbc83453dFba7c16237"

# ABI — defines the contract's functions and events for Web3.py interaction
ABI = [{"inputs":[],"stateMutability":"nonpayable","type":"constructor"},{"inputs":[],"name":"NotAuthorized","type":"error"},{"inputs":[],"name":"NotOwner","type":"error"},{"inputs":[],"name":"PatientAlreadyExist","type":"error"},{"inputs":[],"name":"PatientNotFound","type":"error"},{"anonymous":False,"inputs":[{"indexed":False,"internalType":"uint256","name":"patientId","type":"uint256"},{"indexed":False,"internalType":"uint256","name":"diagnosisResult","type":"uint256"},{"indexed":False,"internalType":"uint256","name":"timestamp","type":"uint256"}],"name":"DiagnosisUploaded","type":"event"},{"anonymous":False,"inputs":[{"indexed":False,"internalType":"uint256","name":"patientId","type":"uint256"},{"indexed":False,"internalType":"address","name":"oldDoctor","type":"address"},{"indexed":False,"internalType":"address","name":"newDoctor","type":"address"}],"name":"DoctorReassigned","type":"event"},{"anonymous":False,"inputs":[{"indexed":False,"internalType":"uint256","name":"patientId","type":"uint256"},{"indexed":False,"internalType":"address","name":"doctorAddress","type":"address"}],"name":"PatientRegistered","type":"event"},{"inputs":[{"internalType":"uint256","name":"_patientId","type":"uint256"},{"internalType":"address","name":"_newDoctor","type":"address"}],"name":"reassignDoctor","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint256","name":"_patientId","type":"uint256"},{"internalType":"address","name":"_doctorAddress","type":"address"}],"name":"registerPatient","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint256","name":"patientId","type":"uint256"},{"internalType":"bytes32","name":"imageHash","type":"bytes32"},{"internalType":"uint256","name":"diagnosisResult","type":"uint256"}],"name":"uploadDiagnosis","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint256","name":"patientId","type":"uint256"},{"internalType":"bytes32","name":"imageHash","type":"bytes32"}],"name":"verifyDiagnosis","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"viewPatients","outputs":[{"internalType":"uint256[]","name":"","type":"uint256[]"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint256","name":"_patientId","type":"uint256"}],"name":"viewRecords","outputs":[{"components":[{"internalType":"bytes32","name":"diagnosisImageHash","type":"bytes32"},{"internalType":"uint256","name":"diagnosisResultCategory","type":"uint256"},{"internalType":"uint256","name":"patientId","type":"uint256"},{"internalType":"address","name":"doctorId","type":"address"},{"internalType":"uint256","name":"timestamp","type":"uint256"}],"internalType":"struct DRDiagnosisResuls.Diagnosis[]","name":"","type":"tuple[]"}],"stateMutability":"view","type":"function"}]
contract = w3.eth.contract(address=CONTRACT_ADDRESS, abi=ABI)
print("Contract loaded!")

Connected: True
Owner Address: 0x8830342a7937361ac4F4407255c8a98ea9A1D775
Contract loaded!


## Register Patient

In [3]:
# ─── Imports ─────────────────────────────────────────────────────────────────
from faker import Faker
import json
import random
import datetime
import os

# ─── Constants ───────────────────────────────────────────────────────────────

# Path to persistent patient records stored on Google Drive
JSON_PATH = '/content/drive/MyDrive/patients.json'

# Doctor registry — maps doctor name to their wallet address
# Note: In this PoC, both doctors share the owner's address for simplicity.
# In production, each doctor would have a separate wallet and private key.
DOCTORS = {
    "doctor1": owner_address,
    "doctor2": owner_address
}

# ─── Function ────────────────────────────────────────────────────────────────

def create_Fake_Patient():
    """
    Generates a synthetic patient profile using Faker.
    Assigns a random doctor from the DOCTORS registry.
    Loads existing patient data from Drive to auto-increment patient ID.
    Returns the new patient dict and the full patient list.
    """

    # Load existing patients from JSON — create file if it doesn't exist
    if not os.path.exists(JSON_PATH):
        with open(JSON_PATH, 'w') as f:
            json.dump([], f, indent=4)
        data = []
    else:
        with open(JSON_PATH, 'r') as f:
            data = json.load(f)

    # Auto-increment patient ID based on existing records
    pat_new_id = len(data) + 1

    # Randomly assign a doctor from the registry
    doctorName = random.choice(list(DOCTORS.keys()))

    faker = Faker()
    patient = {
        "patientId": pat_new_id,
        "doctorName": doctorName,
        "doctorAddress": DOCTORS[doctorName],
        "name": faker.name(),
        "timestamp": datetime.datetime.now().isoformat()
    }

    print(f"Patient created: {patient}")
    return patient, data

In [7]:
def register_patient(patient, data):
    """
    Registers a new patient on the blockchain via the smart contract.
    On success, saves the patient record to the local JSON store on Drive.

    Args:
        patient (dict): Patient data generated by create_Fake_Patient()
        data (list): Existing patient records loaded from JSON

    Returns:
        bool: True if transaction succeeded, False otherwise
    """

    # Build the transaction — calls registerPatient() on the smart contract
    txn = contract.functions.registerPatient(
        patient["patientId"],
        patient["doctorAddress"]
    )..build_transaction({
    'from': owner_address,
    'nonce': w3.eth.get_transaction_count(owner_address, "pending"),
    'gas': 200000,
    'maxFeePerGas': w3.eth.gas_price * 2,
    'maxPriorityFeePerGas': w3.to_wei(2, 'gwei'),
    'type': '0x2'
})

    # Sign the transaction with the owner's private key
    # Signing produces: raw_transaction (broadcast-ready bytes) + hash
    signed_txn = w3.eth.account.sign_transaction(txn, PRIVATE_KEY)

    # Broadcast the signed transaction to the Sepolia network
    # Returns the transaction hash — same value as signed_txn.hash
    # Difference: this hash is confirmed by the network, signed_txn.hash is local
    txn_hash = w3.eth.send_raw_transaction(signed_txn.raw_transaction)

    # Wait for the transaction to be mined and included in a block
    receipt = w3.eth.wait_for_transaction_receipt(txn_hash, timeout=300)

    print(f'Transaction succeeded: {bool(receipt.status)}')
    print(f'Transaction hash: {txn_hash.hex()}')

    # Only save to JSON if the transaction was successful (status == 1)
    if receipt.status == 1:
        data.append(patient)
        with open(JSON_PATH, 'w') as f:
            json.dump(data, f, indent=4)
        print(f"Patient saved to JSON: {patient}")

    return receipt.status == 1

In [12]:
# ─── Utility: Reset Patient JSON ─────────────────────────────────────────────
# Run this cell only when redeploying the smart contract.
# Deletes the local patient JSON to stay in sync with the fresh contract state.

# import os
# JSON_PATH = '/content/drive/MyDrive/patients.json'
# if os.path.exists(JSON_PATH):
#     os.remove(JSON_PATH)
# print("JSON reset complete.")

# ─── Register New Patient ─────────────────────────────────────────────────────
# Generates a synthetic patient and registers them on-chain.
# Patient is saved to JSON only if the blockchain transaction succeeds.

patient, data = create_Fake_Patient()
register_patient(patient, data)

Patient created: {'patientId': 2, 'doctorName': 'doctor2', 'doctorAddress': '0x8830342a7937361ac4F4407255c8a98ea9A1D775', 'name': 'Joshua Lowe', 'timestamp': '2026-06-29T08:15:26.511621'}
Transaction succeeded: True
Transaction hash: fe25b8e891898f05357907d99174b95bd37f3f17731324bad4f80a4e5d7fa8a8
Patient saved to JSON: {'patientId': 2, 'doctorName': 'doctor2', 'doctorAddress': '0x8830342a7937361ac4F4407255c8a98ea9A1D775', 'name': 'Joshua Lowe', 'timestamp': '2026-06-29T08:15:26.511621'}


True

## Model Prediction & Upload Diagnosis

In [14]:
# ─── Model & Test Data Setup ──────────────────────────────────────────────────

import pandas as pd
import os
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

# Load test images from Drive
TEST_IMAGES_PATH = '/content/drive/MyDrive/aptos2019/test_images/test_images'
TEST_CSV_PATH = '/content/drive/MyDrive/aptos2019/test.csv'

test_images = os.listdir(TEST_IMAGES_PATH)
test_csv = pd.read_csv(TEST_CSV_PATH)
print(f"Total test images: {len(test_images)}")

# Load trained EfficientNetB3 model from Drive
model = tf.keras.models.load_model(
    '/content/drive/MyDrive/dr_model_phase2.h5',
    compile=False
)
print("Model loaded!")

# DR severity class labels
CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']

# Tracks images already assigned to patients — prevents duplicate assignments
used_images = []

# ─── TTA Function ────────────────────────────────────────────────────────────

def predict_with_tta(image_path, model, n_augments=5):
    """
    Runs inference using Test Time Augmentation (TTA).
    Averages predictions across original + n augmented versions.
    Improves recall on minority classes (Severe, Proliferative).
    """
    img_array = img_to_array(load_img(image_path, target_size=(224, 224)))
    predictions = []

    # Original prediction
    orig = preprocess_input(np.expand_dims(img_array, axis=0))
    predictions.append(model.predict(orig, verbose=0))

    # Augmented predictions
    datagen = ImageDataGenerator(
        horizontal_flip=True,
        vertical_flip=True,
        rotation_range=30,
        zoom_range=0.1
    )
    for _ in range(n_augments):
        aug = preprocess_input(
            next(datagen.flow(np.expand_dims(img_array, axis=0)))
        )
        predictions.append(model.predict(aug, verbose=0))

    return int(np.argmax(np.mean(predictions, axis=0)))

# ─── Prediction Function ──────────────────────────────────────────────────────

def predict_diagnosis(test_images, test_csv, model):
    """
    Selects a random unused test image, checks its true label,
    and runs TTA-based DR classification.

    Returns:
        tuple: (image_filename, image_path, predicted_class) or (None, None, None)
    """

    # Filter out already-used images
    available = [img for img in test_images if img not in used_images]
    if not available:
        print("No available images left.")
        return None, None, None

    # Randomly select an unused image
    image_filename = random.choice(available)
    image_path = f'{TEST_IMAGES_PATH}/{image_filename}'
    print(f"Selected image: {image_filename}")

    # True label from test CSV
    image_name = image_filename.replace('.png', '')
    real_label = test_csv[test_csv['id_code'] == image_name]['diagnosis'].values[0]
    print(f"Real label:  {CLASS_NAMES[real_label]}")

    # TTA prediction
    predicted_class = predict_with_tta(image_path, model)
    print(f"Predicted:   {CLASS_NAMES[predicted_class]} (Class {predicted_class})")

    return image_filename, image_path, predicted_class

Total test images: 366
Model loaded!


In [19]:
import hashlib

def upload_diagnosis(patient_id, image_filename, predicted_class, data):
    """
    Uploads a diagnosis record for a patient to the blockchain.
    Computes SHA-256 hash of the retinal image and stores it on-chain
    alongside the predicted DR severity class.
    On success, updates the patient's diagnosis history in the JSON store.

    Args:
        patient_id (int): The patient's unique ID
        image_filename (str): Filename of the retinal image used for diagnosis
        predicted_class (int): DR severity predicted by the model (0-4)
        data (list): Full patient records loaded from JSON

    Returns:
        bool: True if transaction succeeded, False otherwise
    """

    image_path = f'{TEST_IMAGES_PATH}/{image_filename}'

    # Verify patient exists in local JSON store
    patient = next((p for p in data if p["patientId"] == patient_id), None)
    if patient is None:
        print(f"Patient {patient_id} not found in JSON.")
        return False

    # Compute SHA-256 hash of the raw image bytes
    # This hash is stored on-chain as tamper-proof proof of the image used
    with open(image_path, 'rb') as f:
        image_hash = hashlib.sha256(f.read()).hexdigest()
    image_hash_bytes32 = bytes.fromhex(image_hash)

    # Build transaction — calls uploadDiagnosis() on the smart contract
    # Only the assigned doctor can upload (enforced by SC)
    txn = contract.functions.uploadDiagnosis(
        patient["patientId"],
        image_hash_bytes32,
        predicted_class
    ).build_transaction({
        'from': owner_address,
        'nonce': w3.eth.get_transaction_count(owner_address, "pending"),
        'gas': 200000,
        'gasPrice': w3.eth.gas_price * 2,
    })

    # Sign transaction with owner's private key
    signed_txn = w3.eth.account.sign_transaction(txn, PRIVATE_KEY)

    # Broadcast to Sepolia network — returns confirmed transaction hash
    txn_hash = w3.eth.send_raw_transaction(signed_txn.raw_transaction)

    # Wait until transaction is mined (up to 5 minutes)
    receipt = w3.eth.wait_for_transaction_receipt(txn_hash, timeout=300)

    print(f"Transaction succeeded: {bool(receipt.status)}")
    print(f"Transaction hash:      {txn_hash.hex()}")

    # On success — update JSON and mark image as used
    if receipt.status == 1:
        # Mark image as assigned — prevents reassignment to another patient
        used_images.append(image_filename)

        # Append diagnosis to patient's history in JSON
        for p in data:
            if p["patientId"] == patient_id:
                if "diagnoses" not in p:
                    p["diagnoses"] = []
                p["diagnoses"].append({
                    "image_filename": image_filename,
                    "predicted_class": predicted_class,
                    "diagnosis_label": CLASS_NAMES[predicted_class]
                })
                break

        with open(JSON_PATH, 'w') as f:
            json.dump(data, f, indent=4)
        print(f"Diagnosis saved to JSON for patient {patient_id}")

    return receipt.status == 1

In [20]:
# ─── Run Pipeline ────────────────────────────────────────────────────────────
# Step 1: Patient registration is done in the cell above.
# Step 2: Select a random unused test image and run DR classification model.
# Step 3: Upload the diagnosis hash and predicted class to the blockchain.

# Step 2 — Select image and predict DR severity
image_filename, image_path, predicted_class = predict_diagnosis(test_images, test_csv, model)

# Step 3 — Upload diagnosis to blockchain
# Pass the patient ID manually — doctor decides which patient this diagnosis belongs to
upload_diagnosis(2, image_filename, predicted_class, data)

Selected image: f6f7dba7104d.png
Real label:  Mild
Predicted:   Moderate (Class 2)
Transaction succeeded: True
Transaction hash:      a63910ef33f7551ad2b853302230567f7134adcb54638d0c54d93d719e4f4280
Diagnosis saved to JSON for patient 2


True

## View Records & View Patients List

In [34]:
def view_records(patient_id):
    """
    Retrieves and displays all diagnosis records for a given patient.
    Fetches on-chain records from the smart contract and enriches them
    with doctor name from the local JSON store.

    Args:
        patient_id (int): The patient's unique ID
    """

    # Load patient data from JSON to get doctor name
    with open(JSON_PATH, 'r') as f:
        data = json.load(f)

    patient = next((p for p in data if p["patientId"] == patient_id), None)

    # Fetch diagnosis records from blockchain — read-only call, no gas required
    records = contract.functions.viewRecords(patient_id).call({'from': owner_address})

    if not records:
        print(f"No records found for patient {patient_id}")
        return

    print(f"Records for Patient {patient_id}:")
    for i, r in enumerate(records):
        print(f"\n  Diagnosis {i+1}:")
        print(f"  Image Hash:  {r[0].hex()}")
        print(f"  Diagnosis:   {CLASS_NAMES[r[1]]}")
        print(f"  Doctor:      {patient['doctorName'] if patient else r[3]}")
        print(f"  Timestamp:   {r[4]}")

view_records(2)

Records for Patient 2:

  Diagnosis 1:
  Image Hash:  b1166fd4de6fa156cea9550b0bdd883a8ca9daa77c9daf003ad1264562dd7137
  Diagnosis:   Moderate
  Doctor:      doctor1
  Timestamp:   1782721308

  Diagnosis 2:
  Image Hash:  3c8d7e4e7d002cdb111ed6dee4b67fd1e9a44ae6fec4d2b39aacfd9de1671c5f
  Diagnosis:   No DR
  Doctor:      doctor1
  Timestamp:   1782721824


In [22]:
def view_patients():
    """
    Retrieves and displays all patients assigned to the calling doctor.
    Fetches patient IDs from the blockchain and enriches with
    name and doctor info from the local JSON store.
    """

    # Fetch patient IDs assigned to this doctor — read-only call, no gas required
    patient_ids = contract.functions.viewPatients().call({'from': owner_address})

    # Load full patient records from JSON for name and doctor info
    with open(JSON_PATH, 'r') as f:
        data = json.load(f)

    print(f"Patients ({len(patient_ids)} total):")
    for pid in patient_ids:
        patient = next((p for p in data if p["patientId"] == pid), None)
        if patient:
            print(f"  ID: {pid} — {patient['name']} — Doctor: {patient['doctorName']}")

view_patients()

Patients (3 total):
  ID: 1 — Jonathan Parrish — Doctor: doctor1
  ID: 2 — Joshua Lowe — Doctor: doctor2


## Verify Diagnoses on Blockchain

In [25]:
def verify_diagnosis(patient_id, image_filename):
    """
    Verifies the integrity of all diagnosis records for a given patient.
    Recomputes the SHA-256 hash of each image stored in JSON and compares
    it against the hash stored on-chain. A mismatch indicates data tampering.

    Args:
        patient_id (int): The patient's unique ID
    """
    image_path = f'{TEST_IMAGES_PATH}/{image_filename}'
    with open(image_path, 'rb') as f:
        image_hash = bytes.fromhex(hashlib.sha256(f.read()).hexdigest())

    result = contract.functions.verifyDiagnosis(patient_id, image_hash).call({'from': owner_address})

    if result:
        print(f"✅ Verified on-chain!")
    else:
        print(f"❌ Tampered or not found!")

    return result


# JSON se image filename nikalo
with open(JSON_PATH, 'r') as f:
    data = json.load(f)

# Retrieve patient with ID 2, as this patient has diagnoses
patient = next((p for p in data if p["patientId"] == 2), None)

if patient and patient["diagnoses"]:
    image_filename = patient["diagnoses"][0]["image_filename"]
    verify_diagnosis(2, image_filename)
else:
    print(f"Patient 2 not found or has no diagnoses to verify.")

✅ Verified on-chain!


In [33]:
def reassign_doctor(patient_id, new_doctor_name):
    """
    Reassigns a patient to a new doctor on-chain.
    Only the admin (owner) can call this function.
    Args:
        patient_id (int): Patient's unique ID
        new_doctor_name (str): Doctor name from DOCTORS registry e.g. 'doctor1', 'doctor2'
    """
    new_doctor_address = DOCTORS[new_doctor_name]

    txn = contract.functions.reassignDoctor(
        patient_id,
        new_doctor_address
    ).build_transaction({
        'from': owner_address,
        'nonce': w3.eth.get_transaction_count(owner_address, "pending"),
        'gas': 200000,
        'gasPrice': w3.eth.gas_price * 2,
    })
    signed_txn = w3.eth.account.sign_transaction(txn, PRIVATE_KEY)
    txn_hash = w3.eth.send_raw_transaction(signed_txn.raw_transaction)
    receipt = w3.eth.wait_for_transaction_receipt(txn_hash, timeout=300)

    print(f"Transaction succeeded: {bool(receipt.status)}")
    print(f"Transaction hash: {txn_hash.hex()}")

    # JSON update karo
    if receipt.status == 1:
        with open(JSON_PATH, 'r') as f:
            data = json.load(f)
        for p in data:
            if p["patientId"] == patient_id:
                p["doctorName"] = new_doctor_name
                p["doctorAddress"] = new_doctor_address
                break
        with open(JSON_PATH, 'w') as f:
            json.dump(data, f, indent=4)
        print(f"Patient {patient_id} reassigned to {new_doctor_name}")

    return receipt.status == 1

reassign_doctor(2, "doctor1")

Transaction succeeded: True
Transaction hash: 7d2c7713441fa0b2edc844e9cf1d22c26c365bde0299b1882f17d93cb7b3690e
Patient 2 reassigned to doctor1


True

In [28]:
# import json

# JSON_PATH = '/content/drive/MyDrive/patients.json'

# with open(JSON_PATH, 'r') as f:
#     data = json.load(f)

# patient_2 = {
#     "patientId": 2,
#     "doctorName": "doctor2",
#     "doctorAddress": "0x8830342a7937361ac4F4407255c8a98ea9A1D775",
#     "name": "Joshua Lowe",
#     "timestamp": "2026-06-29T08:10:06.353324",
#     "diagnoses": []
# }

# data.append(patient_2)

# with open(JSON_PATH, 'w') as f:
#     json.dump(data, f, indent=4)

# print("Done!")
# print(data)

Done!
[{'patientId': 1, 'doctorName': 'doctor1', 'doctorAddress': '0x8830342a7937361ac4F4407255c8a98ea9A1D775', 'name': 'Jonathan Parrish', 'timestamp': '2026-06-29T08:10:06.353324', 'diagnoses': []}, {'patientId': 2, 'doctorName': 'doctor2', 'doctorAddress': '0x8830342a7937361ac4F4407255c8a98ea9A1D775', 'name': 'Joshua Lowe', 'timestamp': '2026-06-29T08:10:06.353324', 'diagnoses': []}]
